In [1]:
import pandas as pd
from datetime import datetime

Part A:
- creating dataset
- parsing
- validation
- event generator

In [2]:
raw_data = [
    {"Timestamp": "08:00", "Route": "AITU-Campus–Residence", "Bus": "B01", "Passengers": 18, "Speed_kmh": 31, "Status": "ON_ROUTE"},
    {"Timestamp": "08:01", "Route": "AITU-Campus–Residence", "Bus": "B02", "Passengers": 22, "Speed_kmh": 28, "Status": "ON_ROUTE"},
    {"Timestamp": "08:02", "Route": "AITU-Campus–Residence", "Bus": "B01", "Passengers": 21, "Speed_kmh": 29, "Status": "ON_ROUTE"},
    {"Timestamp": "08:03", "Route": "AITU-Campus–Residence", "Bus": "B02", "Passengers": 25, "Speed_kmh": 27, "Status": "ON_ROUTE"},
    {"Timestamp": "08:04", "Route": "AITU-Campus–Residence", "Bus": "B01", "Passengers": 24, "Speed_kmh": 0, "Status": "STOPPED"},
    {"Timestamp": "08:05", "Route": "AITU-Campus–Residence", "Bus": "B02", "Passengers": 26, "Speed_kmh": 30, "Status": "ON_ROUTE"},
    {"Timestamp": "08:06", "Route": "AITU-Campus–Residence", "Bus": "B01", "Passengers": 23, "Speed_kmh": 32, "Status": "ON_ROUTE"},
    {"Timestamp": "08:07", "Route": "AITU-Campus–Residence", "Bus": "B02", "Passengers": 28, "Speed_kmh": 26, "Status": "ON_ROUTE"},
    {"Timestamp": "08:08", "Route": "AITU-Campus–Residence", "Bus": "B01", "Passengers": 27, "Speed_kmh": 25, "Status": "ON_ROUTE"},
    {"Timestamp": "08:09", "Route": "AITU-Campus–Residence", "Bus": "B02", "Passengers": 30, "Speed_kmh": 24, "Status": "ON_ROUTE"}
]

df = pd.DataFrame(raw_data)

In [3]:
def parse_event(event_dict):
    parsed = event_dict.copy()
    
    parsed["Timestamp"] = datetime.strptime(event_dict["Timestamp"], "%H:%M").time()
    parsed["Passengers"] = int(event_dict["Passengers"])
    parsed["Speed_kmh"] = int(event_dict["Speed_kmh"])
    return parsed

In [9]:
def validate_event(event_dict):
    if event_dict["Passengers"] < 0:
        return False, "Passengers cannot be negative"
    if not (0 <= event_dict["Speed_kmh"] <= 120):
        return False, "Speed must be between 0 and 120 km/h"
    if event_dict["Status"] not in ["ON_ROUTE", "STOPPED"]:
        return False, "Invalid status"
    return True, "Valid"

In [10]:
def event_generator(data_list):
    for item in data_list:
        parsed_item = parse_event(item)
        is_valid, reason = validate_event(parsed_item)
        
        yield {
            "event": parsed_item,
            "is_valid": is_valid,
            "reason": reason
        }

In [11]:
#checking
display(df.head())

for gen_output in event_generator(raw_data):
    print(gen_output)

,Timestamp,Route,Bus,Passengers,Speed_kmh,Status
0,08:00,AITU-Campus–Residence,B01,18,31,ON_ROUTE
1,08:01,AITU-Campus–Residence,B02,22,28,ON_ROUTE
2,08:02,AITU-Campus–Residence,B01,21,29,ON_ROUTE
3,08:03,AITU-Campus–Residence,B02,25,27,ON_ROUTE
4,08:04,AITU-Campus–Residence,B01,24,0,STOPPED


{'event': {'Timestamp': datetime.time(8, 0), 'Route': 'AITU-Campus–Residence', 'Bus': 'B01', 'Passengers': 18, 'Speed_kmh': 31, 'Status': 'ON_ROUTE'}, 'is_valid': True, 'reason': 'Valid'}
{'event': {'Timestamp': datetime.time(8, 1), 'Route': 'AITU-Campus–Residence', 'Bus': 'B02', 'Passengers': 22, 'Speed_kmh': 28, 'Status': 'ON_ROUTE'}, 'is_valid': True, 'reason': 'Valid'}
{'event': {'Timestamp': datetime.time(8, 2), 'Route': 'AITU-Campus–Residence', 'Bus': 'B01', 'Passengers': 21, 'Speed_kmh': 29, 'Status': 'ON_ROUTE'}, 'is_valid': True, 'reason': 'Valid'}
{'event': {'Timestamp': datetime.time(8, 3), 'Route': 'AITU-Campus–Residence', 'Bus': 'B02', 'Passengers': 25, 'Speed_kmh': 27, 'Status': 'ON_ROUTE'}, 'is_valid': True, 'reason': 'Valid'}
{'event': {'Timestamp': datetime.time(8, 4), 'Route': 'AITU-Campus–Residence', 'Bus': 'B01', 'Passengers': 24, 'Speed_kmh': 0, 'Status': 'STOPPED'}, 'is_valid': True, 'reason': 'Valid'}
{'event': {'Timestamp': datetime.time(8, 5), 'Route': 'AITU-Ca

Part B
Create a generator that simulates arrival of events in chronological order. Do not load the whole stream inside the processing function.

In [16]:
def stream_simulator(data_list):
    """Yields events one by one in chronological order."""
    for result in event_generator(data_list):
        yield result


def process_stream(stream):
    """Processes the stream one event at a time."""
    for item in stream:
        event = item["event"]
        
        if item["is_valid"]:
            print(f"[{event['Timestamp']}] Bus {event['Bus']}: Processed - Valid")
        else:
            print(f"[{event['Timestamp']}] Bus {event['Bus']}: Rejected - {item['reason']}")

process_stream(stream_simulator(raw_data))

[08:00:00] Bus B01: Processed - Valid
[08:01:00] Bus B02: Processed - Valid
[08:02:00] Bus B01: Processed - Valid
[08:03:00] Bus B02: Processed - Valid
[08:04:00] Bus B01: Processed - Valid
[08:05:00] Bus B02: Processed - Valid
[08:06:00] Bus B01: Processed - Valid
[08:07:00] Bus B02: Processed - Valid
[08:08:00] Bus B01: Processed - Valid
[08:09:00] Bus B02: Processed - Valid
